### Web scrapping from Business Day

In [1]:
# Web-scrapping libraries
import requests 
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

In [24]:
# List of URL retrieved

sa_articles_url = [
    "https://www.businessday.co.za/economy/2025-11-10-sanlam-sanparks-smme-fund-a-proven-model-for-scalable-impact/",
    "https://www.businessday.co.za/bd/economy/2025-07-28-sa-borrows-500m-from-germany-to-advance-just-energy-transition/",
    "https://www.businessday.co.za/bd/economy/2025-07-17-gambling-poses-long-term-threat-to-sa-economy-warns-old-mutual/",
    "https://www.businessday.co.za/bd/economy/2025-05-04-economic-week-ahead-eskom-announces-plans-for-winter-power-supply/",
    "https://www.businessday.co.za/bd/national/2025-03-24-sa-in-r67bn-steel-tariff-review/",
    "https://www.businessday.co.za/news/science-and-environment/2025-12-08-code-red-climate-warnings-ethekwini-is-a-sitting-duck-for-next-disaster/",
    "https://www.businessday.co.za/news/health/2025-11-24-gender-based-violence-is-officially-a-disaster-but-will-that-bring-real-change/",
    "https://www.businessday.co.za/bd/national/2025-09-04-five-people-to-stand-trial-for-jagersfontein-dam-disaster-in-2022/",
    "https://www.businessday.co.za/bd/national/2025-06-03-state-of-disaster-sought-as-foot-and-mouth-outbreak-worsens-in-kzn/",
    "https://www.businessday.co.za/bd/national/2024-09-30-sasria-looks-to-extend-cover-to-natural-disasters-and-climate-risks/",
    "https://www.businessday.co.za/bd/national/2025-08-04-treasury-looks-to-private-sector-for-disaster-insurance/",
    "https://www.businessday.co.za/bd/national/2024-10-03-climate-change-damage-amounting-to-billions-hits-kzn-and-western-cape/", 
    "https://mg.co.za/thought-leader/opinion/2024-05-15-the-overlooked-potential-of-basic-income-in-addressing-socio-economic-crises/",
    "https://mg.co.za/columns/2025-06-04-environmental-injustice-is-becoming-the-new-normal-in-sa-we-must-resist-it/",
    "https://mg.co.za/news/2025-05-10-policies-skewed-against-women/",
    "https://mg.co.za/news/2021-07-07-civil-society-coalition-condemns-south-africas-report-to-the-un-on-inequality/"
]

In [25]:
# Create empty dataframe
sa_articles_df = pd.DataFrame(columns=['Title', 'Date', 'Text'])

for sa_article in sa_articles_url:
    response = requests.get(sa_article)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')
        
        # Extract title and date of article
        title = soup.find('h1').get_text(strip = True) if soup.find('h1') else None
        date = soup.find('time').get_text(strip = True) if soup.find('time') else None
        
        # Extract paragraph of article
        word_lst = [p.get_text(strip = True) for p in soup.find_all('p')]
        paragraph = " ".join(word_lst)
        
        # Add row entries
        sa_articles_df.loc[len(sa_articles_df)] = [title, date, paragraph]

In [27]:
sa_articles_df.to_csv("sa_articles.csv")

### Logistic regression

In [30]:
# NLP libraries
import re
import nltk
from nltk.corpus import stopwords
nltk.download('wordnet', quiet = True)
nltk.download('stopwords', quiet = True)
nltk.download('punkt_tab', quiet = True)

# Logistic regression libraries
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# Retrieving recommendations
dataframe = pd.read_csv('llm_recommendations.csv')

# Clean text
stop_words = set(stopwords.words('english'))

def clean_txt(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [word for word in words not in stop_words]
    return " ".join(tokens)
df['clean_txt'] = df['article'].apply(clean_txt)

# Vectorizing
vectorizer = TfidVectorizer(max_features = 5000)

# Defining input feature 
X = vectorizer.fit_transform(df['clean_txt'])

# Defining target variable
dataframe['target'] = LabelEncoder().fit_transform(dataframe['region'])
y = dataframe['target']

In [26]:
# Train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 23)

# Initial logistic regression model
log_model1 = LogisticRegression(random_state = 26)
log_model1.fit(X_train, y_train)

# Calculate accuracy score
accuracy_score1 = accuracy_score(y_test, log_model1.predict(X_test))

# Return accuracy
print(f'The accuracy score is without hyperparameter tuning {round(accuracy_score1 * 100, 2)}%')

The accuracy score is without hyperparameter tuning 75.0%


In [27]:
# Grid search for best logistic parameters

# parameter grid 
param_grid = [{
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'solver': ['lbfgs', 'newton-cg', 'liblinear', 'sag'],
    'max_iter': [100, 1000, 2500, 5000]
}]

# library
from sklearn.model_selection import GridSearchCV
log_model2_params = GridSearchCV(log_model1, param_grid = param_grid, cv = 3, verbose = True, n_jobs = -1)

In [28]:
# best parameters
best_log_model = log_model2_params.fit(X_train, y_train)
best_log_model.best_estimator_
print(f'The accuracy with hyperparameter tuning {best_log_model.score(X_test,y_test)*100:.2f}%')

Fitting 3 folds for each of 64 candidates, totalling 192 fits


/opt/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
132 fits failed out of a total of 192.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
  

LogisticRegression(penalty='l1', random_state=26, solver='liblinear')